# F6-svd-spectral — Session 5: Synthesis Capstone

*One class session, roughly 85 minutes — the closing session of the
second sitting, and of the unit. Prerequisites: Sessions 1–4, all of
them; this session is where the pieces run as one machine.*

**This session:** the full chain on one realistic seeded dataset — a
$(200, 40)$ matrix of measurement records:
$W \to S = WW^{\mathsf T} \to$ SVD $\to$ **spectral decomposition
recovered from the SVD** (full form, zero-padded — verified by the
unit's pinned *invariant* checks, and a demonstration of why naive
column-wise checks must fail here) $\to$ the derived identity
$\lVert S \rVert_F = \sqrt{\sum_i \sigma_i^4}$ $\to$ the rank-$r$
sweep with its error curve and budget reads.
Plus a fully worked normal-form MC, the unit's **Exam Connections**,
and **Going Deeper** pointers.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804

## 1. The Data and the Plan

The dataset: $200$ measurement records over $40$ channels — a tall
$(200, 40)$ table $W$, synthetic and seeded, built the way structured
real-world tables tend to be: a few strong underlying directions
(here five, with planted strengths $40, 30, 22, 16, 12$ — deliberately
well separated) plus low-level noise on every entry.

The capstone plan, each arrow a session:

$$W \;\xrightarrow{\text{S3}}\; U, \sigma, V^{\mathsf T}
\;\xrightarrow{\text{S3 bridge}}\; \text{spectral } S = WW^{\mathsf T}
\;\xrightarrow{\text{S2 invariants}}\; \text{verified}
\;\xrightarrow{\text{S4}}\; W_r \text{ sweep, budgets}$$

In [ ]:
rng = np.random.default_rng(SEED)
n, d = 200, 40
strengths = np.array([40., 30., 22., 16., 12.])

Uraw = rng.normal(0, 1, (n, 5))
Vraw = rng.normal(0, 1, (d, 5))
Un = Uraw / np.sqrt((Uraw**2).sum(axis=0))      # unit columns
Vn = Vraw / np.sqrt((Vraw**2).sum(axis=0))
W = (Un * strengths) @ Vn.T + 0.20 * rng.normal(0, 1, (n, d))

print("W shape:", W.shape)
print("||W||_F:", np.linalg.norm(W))

### Checkpoint 1

1. Before any SVD: how many nonzero singular values can $W$ have at
   most, and — given the construction — roughly how many *large* ones
   do you expect?
2. $S = WW^{\mathsf T}$ will be $(200, 200)$.
   From the bridge (Session 3), how many of its 200 eigenvalues do you
   expect to be (a) large, (b) small-but-nonzero, (c) exactly zero?
   *(Careful — (c) is a trick: the noise term matters.)*

## 2. The Spectrum

One thin SVD; the singular values tell the dataset's story before
anything else is computed.

In [ ]:
U, s, Vt = np.linalg.svd(W, full_matrices=False)
print("top 8 sigma :", np.round(s[:8], 4))
print("last 5 sigma:", np.round(s[-5:], 4))

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(np.arange(1, d + 1), s, marker="o", ms=3)
ax.set_xlabel("index i")
ax.set_ylabel("sigma_i")
ax.set_title("singular value spectrum: 5 planted directions + noise floor")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Reading it: five singular values sit almost exactly on the planted
strengths ($\approx 40.3,\ 30.4,\ 21.5,\ 16.7,\ 11.7$), then the
spectrum cliffs to a slowly decaying noise floor ($\approx 3.9$ down
to $\approx 1.7$).
The separation between $\sigma_5 \approx 11.7$ and
$\sigma_6 \approx 3.9$ (a factor of $\sim 3$) matters beyond
aesthetics: the top of the spectrum is **distinct and well separated**,
which is exactly the precondition under which Session 2's sign pin
allows individual-eigenvector comparisons in the checks below.

### Checkpoint 2

1. The noise floor is nonzero, so what is $\operatorname{rank}(W)$
   here — and why does the answer differ from "5", the honest count of
   *structural* directions?
2. From the printed spectrum alone: will the relative rank-5 error be
   closer to 3% or 30%?
   (Estimate $\sqrt{\sum_{i>5}\sigma_i^2}$ against
   $\lVert W \rVert_F \approx 60.8$ — a rough mental bound from "35
   noise values of size $\sim 2$–$4$" suffices.)

## 3. Spectral-from-SVD: the Full Form, Verified by Invariants

Now the bridge, in production.
We want the complete spectral decomposition of
$S = WW^{\mathsf T}\ (200, 200)$ *without ever calling `eigh` on it* —
from the SVD of $W$ alone (Session 3, FULL form):

$$Q = U_{\text{full}}\ (200, 200), \qquad
\lambda = (\sigma_1^2, \dots, \sigma_{40}^2,
\underbrace{0, \dots, 0}_{160}).$$

**How to verify a claimed decomposition — the pinned invariant trio.**
The zero eigenvalue has multiplicity $160$: within that degenerate
block, *any* orthonormal basis of the null space is correct, every
route picks its own, and two valid answers can disagree by an
arbitrary rotation of 160-dimensional space — differences that
sign-fixing cannot reconcile even in principle.
So the capstone contract never compares degenerate eigenvectors
column-by-column; it checks properties that every correct answer must
share:

1. **Reconstruction:** $\lVert Q \Lambda Q^{\mathsf T} - S \rVert$
   small (we use the max-abs entry gap).
2. **Eigen-equation residuals** for the top (nonzero, separated)
   pairs: $\max_i \lVert S q_i - \lambda_i q_i \rVert_\infty$ small.
3. **Top-$k$ subspace projectors:**
   $\lVert Q_k Q_k^{\mathsf T} - \hat U_k \hat U_k^{\mathsf T}
   \rVert$ small, comparing the *span* of the top-$k$ vectors across
   routes — the object that is actually pinned down when individual
   vectors are only defined up to sign.

In [ ]:
S = W @ W.T
Uf = np.linalg.svd(W, full_matrices=True)[0]      # full (200, 200)
lam_pad = np.concatenate([s**2, np.zeros(n - d)])

# Invariant 1: reconstruction
recon_gap = np.abs(Uf @ np.diag(lam_pad) @ Uf.T - S).max()
print("invariant 1, reconstruction gap:", recon_gap)

# Invariant 2: eigen-equation residuals for the top 5 (separated) pairs
resid = max(np.abs(S @ Uf[:, i] - lam_pad[i] * Uf[:, i]).max()
            for i in range(5))
print("invariant 2, top-5 eigen residual:", resid)

# Invariant 3: top-5 projector vs the eigh route
lam_e, Q_e = np.linalg.eigh(S)
lam_e, Q_e = lam_e[::-1], Q_e[:, ::-1]            # the pinned reorder
k = 5
proj_gap = np.abs(Q_e[:, :k] @ Q_e[:, :k].T - Uf[:, :k] @ Uf[:, :k].T).max()
print("invariant 3, top-5 projector gap:", proj_gap)

# and the eigenvalues themselves agree, including the padded tail:
print("eigenvalue gap (all 200):", np.abs(lam_e - lam_pad).max())

tol = 1e-9
assert recon_gap < tol and resid < tol and proj_gap < tol
print("capstone verification contract satisfied (tol = 1e-9)")

All three invariants at machine precision (the absolute gaps here sit
at $\sim 10^{-13}$ on eigenvalues of size $\sim 10^3$ — scale-aware
reading, Session 2's footnote).

**Why not just compare columns?** Watch a naive check fail on a
perfectly correct answer.
Both routes below are *right*; deep in the zero block their basis
choices differ by more than signs, so even the sign-fix cannot save an
entrywise comparison:

In [ ]:
def signfix(Q):
    # the Session 2 pin: flip columns so the largest-|entry| entry is +
    j = np.abs(Q).argmax(axis=0)
    return Q * np.sign(Q[j, np.arange(Q.shape[1])])


# A separated top eigenvector: sign-fix reconciles the routes...
top_gap = np.abs(signfix(Q_e[:, :1]) - signfix(Uf[:, :1])).max()
print("column 0   (lambda ~ 1624, separated): gap after sign-fix =", top_gap)

# ...but a column deep inside the 160-fold zero block does NOT match:
deep_gap = np.abs(signfix(Q_e[:, 100:101]) - signfix(Uf[:, 100:101])).max()
print("column 100 (lambda = 0, degenerate) : gap after sign-fix =", deep_gap)
print("yet BOTH routes satisfy every invariant -> the check was wrong,",
      "not the answers")

This is the exam-relevant discipline: **match the check to what is
actually determined.**
Individually determined objects (separated eigenpairs, up to sign) may
be compared with the sign-fix; collectively determined objects
(degenerate blocks, subspaces) get invariant checks — reconstruction,
residuals, projectors.

### Checkpoint 3

1. Why is the *top-5 projector* $Q_5 Q_5^{\mathsf T}$ well defined —
   identical across all correct routes — even though each individual
   $q_i$ carries sign freedom?
   (What does flipping $q_i \to -q_i$ do to $q_i q_i^{\mathsf T}$?)
2. The eigenvalue comparison needed no sign-fix and no projectors.
   Why are eigenvalues immune to all of this freedom?
3. Suppose the check `proj_gap` were computed with $k = 100$ instead of
   $k = 5$… would it still pass?
   Reason from what the top-100 span contains (careful: is
   $\lambda_{100}$ separated from $\lambda_{101}$?).

## 4. A Derived Bonus: $\lVert S \rVert_F = \sqrt{\sum_i \sigma_i^4}$

How big is $S = WW^{\mathsf T}$ in Frobenius norm — *without building
$S$*?
Derivation, entirely from this unit's own results, in the generic
register (any tall $W$, any spectrum):

1. $S$ is symmetric PSD with spectral form
   $S = Q \Lambda Q^{\mathsf T}$, eigenvalues
   $\lambda_i = \sigma_i^2 \ge 0$ zero-padded (the bridge, full form).
2. A PSD spectral form **is already an SVD of $S$**: $U = V = Q$
   orthonormal, diagonal middle non-negative (sort descending — done,
   since $\sigma$'s are sorted).
   Hence *the singular values of $S$ are its eigenvalues*
   $\sigma_i(S) = \lambda_i = \sigma_i^2$.
3. Apply Session 4's master identity **to $S$ itself**:
   $$\lVert S \rVert_F^2 = \sum_i \sigma_i(S)^2
   = \sum_i (\sigma_i^2)^2 = \sum_i \sigma_i^4
   \;\;\Rightarrow\;\;
   \lVert S \rVert_F = \sqrt{\textstyle\sum_i \sigma_i^4}.$$
   (The padded zeros contribute nothing.)

The same three moves answer any "norm of a Gram matrix from the
spectrum of its factor" question — a register the exam's
linear-algebra arc is fond of.
Note the fourth powers: big singular values dominate $S$'s norm far
more than $W$'s.

In [ ]:
lhs = np.linalg.norm(S)               # direct, builds on the (200,200) matrix
rhs = np.sqrt((s**4).sum())           # from W's spectrum alone
print("||S||_F direct        :", lhs)
print("sqrt(sum sigma_i^4)   :", rhs)
print("gap:", abs(lhs - rhs))
print("compare ||W||_F^2 =", np.linalg.norm(W)**2, " (not the same thing!)")

### Checkpoint 4

1. $\sigma(W) = (3, 2, 1)$ for some tall $W$.
   Compute $\lVert W \rVert_F$ and $\lVert S \rVert_F$ by hand.
2. In step 2 of the derivation, which hypothesis about $\lambda$ makes
   "spectral form = SVD" work — and what would go wrong for a
   symmetric matrix with a negative eigenvalue?
3. Why do the $160$ padded zeros drop out of the final formula?

## 5. The Rank-$r$ Sweep

Session 4's machinery, at capstone scale: the identity gives the whole
error curve from the spectrum; we cross-check it against directly
built truncations at a few sample ranks (never trust a formula you
haven't collided with reality at least once).

In [ ]:
fro = np.linalg.norm(W)
rs = np.arange(0, d + 1)
rel = np.array([np.sqrt((s[r:]**2).sum()) for r in rs]) / fro

# cross-check the identity at a few sample ranks by direct construction
for r in [1, 5, 20]:
    Wr = U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]
    direct = np.linalg.norm(W - Wr) / fro
    print(f"r = {r:2d}: identity {rel[r]:.6f}  direct {direct:.6f}  "
          f"gap {abs(rel[r] - direct):.1e}")

captured = np.cumsum(s**2) / (s**2).sum()   # kept-content fraction

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(rs, rel, marker="o", ms=3)
axes[0].axhline(0.30, ls="--", lw=1)
axes[0].set_xlabel("r")
axes[0].set_ylabel("relative error")
axes[0].set_title("relative error vs r (dashed: 30% budget)")
axes[1].plot(np.arange(1, d + 1), captured, marker="o", ms=3)
axes[1].axhline(0.90, ls="--", lw=1)
axes[1].set_xlabel("r")
axes[1].set_ylabel("captured fraction of ||W||_F^2")
axes[1].set_title("kept content vs r (dashed: 90%)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for r in [3, 4, 5, 6]:
    print(f"r = {r}: relative error {rel[r]:.4f}   "
          f"captured {captured[r - 1]:.4f}")

The elbow at $r = 5$ is unmistakable in both views — the five planted
directions, recovered by pure numerics.
Past the elbow, each extra atom shaves one noise parcel: visible
diminishing returns.

### Checkpoint 5

1. From the printed table: the smallest $r$ with relative error
   $\le 30\%$, and the smallest $r$ capturing $\ge 90\%$ of the
   content.
   Coincidence that both budgets land on the same $r$?
   (Relate $\text{rel}^2$ and captured fraction *exactly*.)
2. State the exact relationship: if the captured fraction at $r$ is
   $c_r$, what is the relative error at $r$?
   Verify on the $r = 5$ row.

## 6. Budget Reads and the Storage Payoff

The two standard budget questions, answered off the curve:

- **Error budget** ("relative error at most 30%"): smallest such $r$
  is $\mathbf{5}$ (relative error $0.272$; $r = 4$ gives $0.333$ —
  too big).
- **Content budget** ("keep at least 90% of
  $\lVert W \rVert_F^2$"): smallest such $r$ is $\mathbf{5}$ again
  (captured $0.926$; $r = 4$ captures $0.889$).

And the payoff for stopping at $r = 5$: storage
$5 \cdot (200 + 40 + 1) = 1205$ floats versus $8000$ — the dataset's
structural content in **15%** of the space, at a fully accounted-for
27% relative error.

In [ ]:
r_err = int(np.argmax(rel <= 0.30))
r_cap = int(np.argmax(captured >= 0.90)) + 1     # captured is indexed from r=1
print("smallest r with rel error <= 30%:", r_err)
print("smallest r with captured >= 90% :", r_cap)
print("storage at r=5:", 5 * (n + d + 1), "of", n * d,
      f"floats ({5 * (n + d + 1) / (n * d):.1%})")

### Checkpoint 6

1. Why is `captured` indexed "from $r = 1$" (note the `+ 1`) while
   `rel` is indexed from $r = 0$?
   What are `captured[0]` and `rel[0]`, in words?
2. The two budget answers used `argmax` on a boolean array.
   Why does `np.argmax` return the *first* qualifying index, and what
   silent wrong answer does it return if *no* index qualifies —
   which check would you add in production code?

## 7. Worked Exam-Style Example: Budget Pick, Normal Form

---

**Worked exam-style example 5 (multiple choice, numeric normal form).**

> A $(9, 6)$ matrix $W$ has singular values
> $\sigma = (12, 6, 4, 3, 2, 1)$.
> Let $r^\*$ be the smallest $r$ such that the rank-$r$ truncation has
> **relative** Frobenius error at most $25\%$, and set
> $s = 10 - 3r^\*$.
> Your answer can be written as $s = \pm m$ with $m$ a non-negative
> integer.
> What is the value of $m + 1$ if $s \ge 0$, or $m + 2$ if $s < 0$?
>
> A. 2  B. 4  C. 5  D. 7  E. 10
>
> Reasoning is not required.

*Solution, step by step.*

1. **Total content.** $\lVert W \rVert_F^2 = 144 + 36 + 16 + 9 + 4 + 1
   = 210$.
2. **Relative error per $r$** — identity, then divide:
   $\text{rel}(r) = \sqrt{\sum_{i>r}\sigma_i^2 / 210}$.
   $r = 2$: $\sqrt{30/210} = 0.378$; $r = 3$: $\sqrt{14/210} = 0.258$;
   $r = 4$: $\sqrt{5/210} = 0.154$.
3. **Threshold.** First $r$ at or below $0.25$: $r^\* = 4$
   ($r = 3$ *just* misses at $0.258$ — resist eyeballing;
   $14/210 = 0.0667 > 0.0625 = 0.25^2$ settles it in fractions).
4. **Target.** $s = 10 - 12 = -2$.
5. **Normal form.** $m = 2$, branch $s < 0$.
6. **Decode.** $m + 2 = 4$ → **B**.

Distractor forensics: conclude $r^\* = 3$ (the eyeball trap in step 3)
and $s = 1$ decodes to $m + 1 = 2$ = **A**.
Apply the budget to *squared* relative error
($\sum_{i>r}\sigma_i^2/210 \le 0.25$ — mixing Pitfall 1 into the
threshold) and $r^\* = 2$, $s = 4 \to 5$ = **C**.
Off-by-one to $r^\* = 5$: $s = -5 \to 7$ = **D**.
The step-3 fraction comparison is where this item is won or lost.

In [ ]:
sig = np.array([12., 6., 4., 3., 2., 1.])
tot = (sig**2).sum()
rel_mc = np.sqrt(np.array([(sig[r:]**2).sum() for r in range(7)]) / tot)
print("relative errors r=0..6:", np.round(rel_mc, 4))
r_star = int(np.argmax(rel_mc <= 0.25))
s_val = 10 - 3 * r_star
m = abs(s_val)
decoded = m + 1 if s_val >= 0 else m + 2
print("r* =", r_star, " s =", s_val, " decoded =", decoded, "-> option B")

### Checkpoint 7

1. Same data, but the budget is stated as "capture at least $95\%$ of
   $\lVert W \rVert_F^2$".
   Find the new $r^\*$ and decode with the same rule.
2. In step 3, why is settling the comparison "in fractions"
   ($14/210$ vs $0.25^2$) more reliable than comparing the rounded
   decimals $0.258$ vs $0.25$?

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **linear-algebra cluster is the paper's largest** (10 sub-parts,
  60 of 300 points in r1-2026), and its advanced back half is
  *exactly* this unit: SVD calls, singular values of a data matrix,
  spectral/eigen readings of similarity (Gram) matrices, low-rank
  approximation with error accounting.
  The analysis flags SVD/spectral/Eckart–Young as the material
  "requiring dedicated curriculum beyond foundation scope" — this unit
  is that curriculum.
- The paper's dominant **multi-part arc** (90 points in r1-2026)
  builds a single narrative where an SVD feeds a spectral step, norm
  identities are derived in terms of $\sigma$, and an error-vs-rank
  curve is plotted and read — precisely this session's pipeline
  texture, with later parts consuming earlier parts' results.
  Derivation sub-parts there are "reasoning required"; the
  $\lVert S \rVert_F = \sqrt{\sum \sigma^4}$-style identity (Section
  4) is the *kind* of derivation that arc rewards.
- The **NumPy implementation cluster** (8 sub-parts, 55 points)
  scopes tool legality per problem — some parts ban `np.linalg` to
  force hand routes, others hint the SVD call — the convention Session
  3 taught; expect zero-points clauses and exact identifier contracts,
  as in this unit's practice.
- **Multiple-choice items** use exactly five options A–E with numeric
  answers in the normal-form register (sign convention + decode rule)
  — worked here in Sessions 1, 4, and 5.

If your budget is tight, the highest-yield hour in this unit is:
the bridge in both forms (thin/full), the error identity with a budget
read, and one `eigh`-reorder + invariant-verification rep.

## Going Deeper

Optional forward pointers along the course map — nothing here is
needed for this unit's practice:

- **`C9-dimensionality-reduction`**: the direct consumer of this unit.
  There, the data matrix's rows are *centered* (mean subtracted) and
  the SVD's top right-singular directions become principal components:
  the directions of maximal variance in real datasets, with
  variance-explained ratios playing the role our captured-content
  curve played here — plus honest lessons about what 2-D projections
  can and cannot show.
  The bridge and the sign conventions pinned in this unit are used
  there by name.
- **`C8-embeddings`** (if you follow the applied track): the
  similarity matrices built from stacked unit-normalized rows are
  exactly the Gram matrices whose spectral structure this unit taught —
  and truncating their factors is how large similarity systems are
  compressed in practice.

*(End of the double unit. `review.ipynb` consolidates all five
sessions; the practice sets are mapped in the unit overview.)*

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. At most $\min(200, 40) = 40$; expect $5$ large ones (the planted
   strengths), with the rest forming a noise floor.
2. (a) $5$ large ($\approx$ strengths$^2$); (b) $35$ small-but-nonzero
   — the noise makes $W$ full-rank ($40$ nonzero $\sigma$'s), so all
   $40$ bridge eigenvalues $\sigma_i^2$ are positive; (c) exactly zero:
   the remaining $200 - 40 = 160$ — a shape fact
   ($\operatorname{rank}(S) \le 40$), noise or no noise.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $\operatorname{rank}(W) = 40$ (every $\sigma_i > 0$; the noise
   floor never reaches zero).
   "5" counts the *planted structural* directions — rank is a
   yes/no-nonzero count and doesn't distinguish structure from noise;
   that distinction is the truncation's job.
2. Closer to 30%: the dropped content is
   $\sim 35 \times (2\text{–}4)^2 \approx 300$, so the error is
   $\sim \sqrt{300} \approx 17$, i.e. $17/60.8 \approx 28\%$
   — matching the computed $27.2\%$.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Flipping $q_i \to -q_i$ leaves $q_i q_i^{\mathsf T} =
   (-q_i)(-q_i)^{\mathsf T}$ unchanged, so the projector — a sum of
   such atoms — is sign-blind; and as long as
   $\lambda_5 > \lambda_6$, the top-5 *span* itself is uniquely
   determined, so all correct routes produce the same projector.
2. Eigenvalues are plain numbers attached to the matrix — no basis
   choice, no scaling freedom is involved in them; only eigen*vectors*
   carry conventions.
3. Yes, it would still pass — but for a subtler reason: the top-100
   span contains all 40 nonzero-eigenvalue directions *plus 60
   directions chosen arbitrarily inside the zero block*, and
   $\lambda_{100} = \lambda_{101} = 0$ is NOT separated, so the
   top-100 span is *not* uniquely determined and the two routes could
   legitimately differ.
   The check would compare two arbitrary 60-dimensional slices of the
   same null space — passing or failing unpredictably.
   Projector checks inherit the separation requirement at their cut:
   cut only where $\lambda_k > \lambda_{k+1}$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $\lVert W \rVert_F = \sqrt{9 + 4 + 1} = \sqrt{14} \approx 3.742$;
   $\lVert S \rVert_F = \sqrt{81 + 16 + 1} = \sqrt{98} = 7\sqrt2
   \approx 9.899$.
2. Non-negativity ($\lambda_i \ge 0$): SVD middles must be
   non-negative.
   With a negative eigenvalue, the spectral form is *not* an SVD as
   written — the singular values become $|\lambda_i|$ (fold the sign
   into one vector family), so the identity would read
   $\sum \lambda_i^2$ anyway, but the clean "eigenvalues = singular
   values" step genuinely requires PSD.
3. They enter as $0^2 = 0$ in the sum $\sum_i \sigma_i(S)^2$ —
   zero-content parcels.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Both are $r = 5$.
   Not a coincidence in structure: $\text{rel}(r)^2 = 1 - c_r$
   exactly (dropped fraction = 1 − kept fraction), so the two
   thresholds $0.30$ and $0.90$ ask *related* questions —
   $\text{rel} \le 0.30 \iff c_r \ge 1 - 0.09 = 0.91$, which here
   lands on the same $r$ as $c_r \ge 0.90$ because the elbow is sharp.
   (With thresholds chosen adversarially the answers can differ by an
   atom or two.)
2. $\text{rel}(r) = \sqrt{1 - c_r}$.
   At $r = 5$: $\sqrt{1 - 0.9262} = \sqrt{0.0738} \approx 0.2717$ ✓
   (printed $0.2716$, rounding).

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. `captured` is a cumulative sum of kept parcels, whose first entry
   corresponds to keeping one atom ($r = 1$); `rel`'s first entry is
   the empty truncation ($r = 0$).
   `captured[0]` = fraction of content in the top atom
   ($\sigma_1^2 / \sum \sigma_i^2$); `rel[0]` $= 1$ (dropping
   everything).
2. `argmax` on booleans returns the first `True` because it returns
   the first occurrence of the maximum value (`True` = 1).
   If nothing qualifies, all values are `False` (= 0) and it returns
   **0** — a plausible-looking rank! — so production code adds e.g.
   `assert (rel <= 0.30).any()` before trusting the read.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Captured fractions: $c_r = \sum_{i \le r}\sigma_i^2 / 210$:
   $c_3 = 196/210 = 0.933$, $c_4 = 205/210 = 0.976$.
   First $\ge 0.95$: $r^\* = 4$ — so $s = -2$, decode $m + 2 = 4$,
   option B again (the two budgets happen to agree here).
2. The rounded decimal $0.258$ hides whether the true value cleared
   the threshold before rounding; the fraction comparison
   $14/210$ vs $1/16$ is exact integer arithmetic
   ($14 \cdot 16 = 224 > 210$), immune to display rounding.

</details>